<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>양자화 개념과 FP16(Quantization Concepts & FP16)</strong>에 대해 학습합니다.</br></br>
>수치 정밀도의 종류와 양자화의 원리, FP16 추론의 효과를 학습해봅시다.

</br>

# 양자화 (Quantization)
> 모델의 가중치를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">더 낮은 정밀도의 숫자</mark>로 변환하여 메모리와 연산량을 줄이는 기법입니다.

대형 언어 모델의 추론(inference)은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">막대한 GPU 메모리</mark>를 요구합니다. 예를 들어 7B 모델은 FP32 기준 28GB, FP16 기준 14GB의 메모리가 필요해 일반 GPU(8~12GB)로는 로드 자체가 불가능합니다. 양자화가 필요한 핵심 이유는 세 가지입니다. 첫째, 클라우드 GPU 사용료를 줄이고 응답 속도를 높일 수 있습니다. 둘째, 스마트폰이나 IoT 기기처럼 메모리가 제한된 엣지 디바이스에서 모델을 실행하려면 크기를 줄여야 합니다. 셋째, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">FP32 에서 FP16으로만 바꿔도 메모리가 절반</mark>이 되어 더 큰 배치나 더 긴 시퀀스를 처리할 수 있습니다.

이 내용을 이해하려면 이진수 표현, 부동소수점(FP32 = 부호 1 + 지수 8 + 가수 23비트), GPU 메모리(VRAM), 모델 파라미터 수(7B = 70억 개 가중치) 등의 개념에 대한 기본 이해가 필요합니다.

</br>

## 수치 정밀도 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">형식</th>
      <th style="text-align:center">비트 수</th>
      <th style="text-align:center">메모리 비율</th>
      <th>특징</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">FP32</td><td style="text-align:center">32</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">100%</mark> (기준)</td><td>최고 정밀도, 학습 기본값</td></tr>
    <tr><td style="text-align:center">FP16</td><td style="text-align:center">16</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">50%</mark></td><td>추론에 충분한 정밀도</td></tr>
    <tr><td style="text-align:center">INT8</td><td style="text-align:center">8</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">25%</mark></td><td>정수 양자화, 약간의 정밀도 손실</td></tr>
    <tr><td style="text-align:center">INT4</td><td style="text-align:center">4</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">12.5%</mark></td><td>최대 압축, QLoRA에 활용</td></tr>
  </tbody>
</table>

</br>

## 메모리 계산 예시

In [ ]:
# TODO 1: params=7e9(70억)으로 설정하고 FP32(4바이트), FP16(2바이트), INT8(1바이트), INT4(0.5바이트) 각각의 메모리(GB)를 계산하여 비교 출력해봅시다.

params = 7e9  # 70억 파라미터

fp32_gb = params * 4 / 1e9    # 4 bytes per param
fp16_gb = params * 2 / 1e9    # 2 bytes per param
int8_gb = params * 1 / 1e9    # 1 byte per param
int4_gb = params * 0.5 / 1e9  # 0.5 bytes per param

print(f"{'형식':<6} {'메모리':>8} {'비율':>6}")
print(f"{'FP32':<6} {fp32_gb:>7.1f}GB {100:>5.0f}%")
print(f"{'FP16':<6} {fp16_gb:>7.1f}GB {50:>5.0f}%")
print(f"{'INT8':<6} {int8_gb:>7.1f}GB {25:>5.0f}%")
print(f"{'INT4':<6} {int4_gb:>7.1f}GB {12.5:>5.1f}%")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">형식</th>
      <th style="text-align:center">메모리</th>
      <th style="text-align:center">비율</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">FP32</td><td style="text-align:center">28.0 GB</td><td style="text-align:center">100%</td></tr>
    <tr><td style="text-align:center">FP16</td><td style="text-align:center">14.0 GB</td><td style="text-align:center">50%</td></tr>
    <tr><td style="text-align:center">INT8</td><td style="text-align:center">7.0 GB</td><td style="text-align:center">25%</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">INT4</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">3.5 GB</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">12.5%</mark></td></tr>
  </tbody>
</table>

</br>

# FP16 추론 (Half-Precision Inference)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">torch.float16</mark>으로 모델을 로드하여 메모리를 절반으로 줄입니다.

In [ ]:
# TODO 2: 사전학습된 모델을 FP16 정밀도와 자동 디바이스 매핑으로 로드하고, dtype과 device를 출력해봅시다.

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    "model_name",
    torch_dtype=torch.float16,  # FP16으로 로드
    device_map="auto"
)
print(f"모델 dtype: {next(model.parameters()).dtype}")
print(f"디바이스: {next(model.parameters()).device}")

</br>

## 추론 성능 측정

In [ ]:
# TODO 3: 추론 시간 측정 함수를 정의하여 그래디언트 계산 없이 모델 생성을 num_runs=5회 반복 실행하고, 평균 추론 시간을 측정해봅시다.

import time

def measure_inference(model, tokenizer, prompt, num_runs=5):
    """추론 시간 측정"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    times = []

    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=100)
        times.append(time.time() - start)

    avg_time = sum(times) / len(times)
    print(f"평균 추론 시간: {avg_time:.2f}초 ({num_runs}회 평균)")
    return avg_time

# FP16 추론 시간 측정
fp16_time = measure_inference(model, tokenizer, "AI의 미래는")

💡device_map="auto"
> 모델을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">사용 가능한 GPU에 자동 분배</mark>합니다.</br>
> GPU 메모리가 부족하면 일부를 CPU/디스크에 오프로드합니다.

💡FP16의 한계
> FP16은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 시 수치 불안정</mark>을 유발할 수 있습니다.</br>
> 학습에는 BF16(Brain Float 16)이나 Mixed Precision이 더 안전합니다.